In [ ]:
"""
Spatio-Temporal Graph-Based Power Outage Prediction Model
=========================================================

This model predicts:
1. Power outage risk in Michigan counties using weather and geographic data
2. Recovery time estimation for outages
3. Extreme event detection (≥50,000 customers out)

Features:
- Graph Neural Networks for spatial relationships
- LSTM for temporal patterns
- Weather integration for storm prediction
- Proactive risk assessment
- Apple GPU (MPS) support for M1/M2 Macs
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.utils import add_self_loops
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import pickle
import warnings
warnings.filterwarnings('ignore')

# ============================
# 🔽 DEVICE SETUP FOR APPLE GPU
# ============================

def get_device():
    """Get the best available device (Apple GPU > CUDA > CPU)"""
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple GPU (MPS)")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA GPU (CUDA)")
    else:
        device = torch.device("cpu")
        print("💻 Using CPU")
    
    return device

# ============================
# 1. DATA LOADING AND PREPROCESSING
# ============================

class PowerOutageDataProcessor:
    def __init__(self):
        self.scalers = {}
        self.label_encoders = {}
        self.county_to_idx = {}
        self.idx_to_county = {}

    def load_and_preprocess_data(self, data_path):
        """Load and preprocess the power outage data"""
        print("🔄 Loading and preprocessing data...")

        # Load the optimized dataset
        df = pd.read_csv(data_path)

        # Ensure datetime is properly formatted
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Create additional temporal features
        df['year'] = df['datetime'].dt.year
        df['month'] = df['datetime'].dt.month
        df['day'] = df['datetime'].dt.day
        df['hour'] = df['datetime'].dt.hour
        df['day_of_year'] = df['datetime'].dt.dayofyear
        df['week_of_year'] = df['datetime'].dt.isocalendar().week
        df['is_weekend'] = (df['datetime'].dt.weekday >= 5).astype(int)

        # Create time-based features
        df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24)
        df['sin_day_year'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
        df['cos_day_year'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

        # Extreme event classification (≥50,000 customers out)
        df['is_extreme_event'] = (df['customers_out'] >= 50000).astype(int)

        # Outage severity levels for multi-class prediction
        df['outage_severity_level'] = pd.cut(
            df['customers_out'],
            bins=[0, 100, 1000, 10000, 50000, float('inf')],
            labels=['minimal', 'minor', 'moderate', 'major', 'extreme'],
            include_lowest=True
        )

        # Recovery time categories
        df['recovery_time_category'] = pd.cut(
            df['DOI_hours'],
            bins=[0, 2, 8, 24, 72, float('inf')],
            labels=['quick', 'short', 'medium', 'long', 'extended'],
            include_lowest=True
        )

        # Calculate risk scores
        df['risk_score'] = self._calculate_risk_score(df)

        print(f"✅ Processed {len(df):,} records")
        print(f"📊 Extreme events: {df['is_extreme_event'].sum():,}")
        print(f"📈 Severity distribution:\n{df['outage_severity_level'].value_counts()}")

        return df

    def _calculate_risk_score(self, df):
        """Calculate a comprehensive risk score"""
        # Normalize factors to 0-1 scale
        outage_rate_norm = df['outage_rate'].fillna(0)
        customers_norm = df['customers_out'] / df['Customers']
        doi_norm = df['DOI_hours'] / 72  # Normalize to 72 hours

        # Weather risk factors
        weather_risk = 0
        if 'wind_speed_10m' in df.columns:
            weather_risk += (df['wind_speed_10m'].fillna(0) / 50)  # Normalize to 50 mph
        if 'precipitation' in df.columns:
            weather_risk += (df['precipitation'].fillna(0) / 10)   # Normalize to 10mm
        if 'snowfall' in df.columns:
            weather_risk += (df['snowfall'].fillna(0) / 5)        # Normalize to 5mm

        # Composite risk score
        risk_score = (
            outage_rate_norm * 0.3 +
            customers_norm * 0.3 +
            np.clip(doi_norm, 0, 1) * 0.2 +
            np.clip(weather_risk, 0, 1) * 0.2
        )

        return np.clip(risk_score, 0, 1)

    def create_county_graph(self, df):
        """Create county adjacency graph based on geographic proximity"""
        print("🗺️ Creating county adjacency graph...")

        # Get unique counties and create mapping
        counties = df['fips_code'].unique()
        self.county_to_idx = {county: idx for idx, county in enumerate(counties)}
        self.idx_to_county = {idx: county for county, idx in self.county_to_idx.items()}

        # For simplicity, create adjacency based on geographic distance
        county_coords = df.groupby('fips_code')[['lat', 'lng']].first()

        # Calculate distance matrix
        edge_index = []
        edge_weight = []

        for i, county1 in enumerate(counties):
            for j, county2 in enumerate(counties):
                if i != j:
                    # Calculate geographic distance
                    lat1, lon1 = county_coords.loc[county1, ['lat', 'lng']]
                    lat2, lon2 = county_coords.loc[county2, ['lat', 'lng']]

                    if pd.notna(lat1) and pd.notna(lat2):
                        distance = np.sqrt((lat1 - lat2)**2 + (lon1 - lon2)**2)

                        # Connect counties within ~1 degree (approximately 100km)
                        if distance < 1.0:
                            edge_index.append([i, j])
                            edge_weight.append(1.0 / (1.0 + distance))  # Inverse distance weighting

        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_weight = torch.tensor(edge_weight, dtype=torch.float)

        print(f"✅ Created graph with {len(counties)} nodes and {len(edge_index[0])} edges")

        return edge_index, edge_weight, county_coords

    def prepare_sequences(self, df, sequence_length=24, prediction_horizon=12):
        """Prepare temporal sequences for LSTM"""
        print(f"📊 Preparing temporal sequences (length={sequence_length}, horizon={prediction_horizon})...")

        sequences = []
        targets_regression = []
        targets_classification = []
        targets_recovery = []

        # Sort by county and datetime
        df_sorted = df.sort_values(['fips_code', 'datetime']).reset_index(drop=True)

        # Feature columns for input
        feature_cols = [
            'customers_out', 'outage_rate', 'risk_score',
            'sin_hour', 'cos_hour', 'sin_day_year', 'cos_day_year',
            'is_weekend', 'month', 'day'
        ]

        # Add weather features if available
        weather_cols = ['wind_speed_10m', 'precipitation', 'snowfall', 'temperature_2m']
        for col in weather_cols:
            if col in df.columns:
                feature_cols.append(col)

        # Fill missing values
        for col in feature_cols:
            if col in df.columns:
                df_sorted[col] = df_sorted[col].fillna(df_sorted[col].mean())

        # Create sequences for each county
        for county in df_sorted['fips_code'].unique():
            county_data = df_sorted[df_sorted['fips_code'] == county].copy()

            if len(county_data) < sequence_length + prediction_horizon:
                continue

            for i in range(len(county_data) - sequence_length - prediction_horizon + 1):
                # Input sequence
                sequence = county_data.iloc[i:i+sequence_length][feature_cols].values

                # Target values (future values to predict)
                future_data = county_data.iloc[i+sequence_length:i+sequence_length+prediction_horizon]

                # Regression targets (risk score, customers_out)
                target_reg = np.column_stack([
                    future_data['risk_score'].values,
                    future_data['customers_out'].values / future_data['Customers'].values  # Normalized outage rate
                ])

                # Classification targets (extreme events)
                target_cls = future_data['is_extreme_event'].values

                # Recovery time targets
                target_recovery = future_data['DOI_hours'].values

                sequences.append(sequence)
                targets_regression.append(target_reg)
                targets_classification.append(target_cls)
                targets_recovery.append(target_recovery)

        print(f"✅ Created {len(sequences)} sequences")

        return (np.array(sequences), np.array(targets_regression),
                np.array(targets_classification), np.array(targets_recovery))

# ============================
# 2. SPATIO-TEMPORAL GRAPH NEURAL NETWORK MODEL
# ============================

class SpatialEncoder(nn.Module):
    """Graph Neural Network for spatial relationships between counties"""

    def __init__(self, input_dim, hidden_dim, output_dim, num_heads=4):
        super(SpatialEncoder, self).__init__()

        # Multi-head Graph Attention Network
        self.gat1 = GATConv(input_dim, hidden_dim, heads=num_heads, dropout=0.1, add_self_loops=False)
        self.gat2 = GATConv(hidden_dim * num_heads, hidden_dim, heads=1, dropout=0.1, add_self_loops=False)

        # Graph Convolutional layers
        self.gcn1 = GCNConv(hidden_dim, hidden_dim, add_self_loops=False)
        self.gcn2 = GCNConv(hidden_dim, output_dim, add_self_loops=False)

        self.dropout = nn.Dropout(0.1)
        self.norm1 = nn.LayerNorm(hidden_dim * num_heads)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x, edge_index, edge_weight=None):
        # Add self-loops manually to handle dimension mismatch
        edge_index, edge_weight = add_self_loops(edge_index, edge_weight, num_nodes=x.size(0))

        # Multi-head attention
        x = self.gat1(x, edge_index)
        x = F.relu(self.norm1(x))
        x = self.dropout(x)

        x = self.gat2(x, edge_index)
        x = F.relu(self.norm2(x))
        x = self.dropout(x)

        # Graph convolution
        x = self.gcn1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.gcn2(x, edge_index, edge_weight)

        return x

class TemporalEncoder(nn.Module):
    """LSTM for temporal pattern modeling"""

    def __init__(self, input_dim, hidden_dim, num_layers=2):
        super(TemporalEncoder, self).__init__()

        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            batch_first=True, dropout=0.1, bidirectional=True
        )

        self.attention = nn.MultiheadAttention(
            hidden_dim * 2, num_heads=8, dropout=0.1, batch_first=True
        )

        self.norm = nn.LayerNorm(hidden_dim * 2)

    def forward(self, x):
        # LSTM encoding
        lstm_out, (hidden, cell) = self.lstm(x)

        # Self-attention
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        attn_out = self.norm(attn_out + lstm_out)  # Residual connection

        return attn_out, hidden

class SpatioTemporalOutagePredictor(nn.Module):
    """Complete spatio-temporal model for outage prediction"""

    def __init__(self, input_dim, spatial_dim, temporal_dim, output_dim,
                 num_counties, prediction_horizon=12):
        super(SpatioTemporalOutagePredictor, self).__init__()

        self.input_dim = input_dim
        self.prediction_horizon = prediction_horizon
        self.num_counties = num_counties

        # Spatial encoding
        self.spatial_encoder = SpatialEncoder(input_dim, spatial_dim, spatial_dim)

        # Temporal encoding
        self.temporal_encoder = TemporalEncoder(input_dim, temporal_dim)

        # Feature fusion
        fusion_dim = spatial_dim + temporal_dim * 2
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(fusion_dim // 2, fusion_dim // 4),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        # Prediction heads
        pred_input_dim = fusion_dim // 4

        # Regression head (risk score, outage rate)
        self.regression_head = nn.Sequential(
            nn.Linear(pred_input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, prediction_horizon * 2)  # 2 regression targets
        )

        # Classification head (extreme events)
        self.classification_head = nn.Sequential(
            nn.Linear(pred_input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, prediction_horizon)  # Binary classification per time step
        )

        # Recovery time prediction head
        self.recovery_head = nn.Sequential(
            nn.Linear(pred_input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, prediction_horizon)  # Recovery time per time step
        )

    def forward(self, temporal_features, spatial_features, edge_index, edge_weight=None):
        batch_size, seq_len, feature_dim = temporal_features.shape

        # Temporal encoding
        temporal_encoded, _ = self.temporal_encoder(temporal_features)
        temporal_summary = temporal_encoded.mean(dim=1)  # Global temporal representation

        # For spatial encoding, we need to handle the graph structure properly
        # Use the mean spatial features across time and batch for graph processing
        spatial_summary = spatial_features.mean(dim=(0, 1))  # Shape: [feature_dim]

        # Create a batch of spatial features for each county
        num_counties = edge_index.max().item() + 1
        spatial_input = spatial_summary.unsqueeze(0).repeat(num_counties, 1)  # [num_counties, feature_dim]

        # Spatial encoding
        spatial_encoded = self.spatial_encoder(spatial_input, edge_index, edge_weight)

        # Take mean across counties to get a single representation per batch item
        spatial_representation = spatial_encoded.mean(dim=0).unsqueeze(0).repeat(batch_size, 1)

        # Fusion
        combined = torch.cat([spatial_representation, temporal_summary], dim=1)
        fused = self.fusion(combined)

        # Predictions
        regression_pred = self.regression_head(fused)
        classification_pred = self.classification_head(fused)
        recovery_pred = self.recovery_head(fused)

        # Reshape outputs
        regression_pred = regression_pred.view(batch_size, self.prediction_horizon, 2)
        classification_pred = classification_pred.view(batch_size, self.prediction_horizon)
        recovery_pred = recovery_pred.view(batch_size, self.prediction_horizon)

        return regression_pred, classification_pred, recovery_pred

# ============================
# 3. TRAINING AND EVALUATION
# ============================

class OutageModelTrainer:
    def __init__(self, model, device):
        self.model = model.to(device)
        self.device = device
        self.training_history = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': []
        }

    def train_model(self, train_loader, val_loader, epochs=100, lr=0.001):
        """Train the spatio-temporal model"""
        print(f"🚀 Starting training for {epochs} epochs on {self.device}...")

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=10
        )

        # Loss functions
        mse_loss = nn.MSELoss()
        bce_loss = nn.BCEWithLogitsLoss()

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0
            train_acc = 0
            num_batches = 0

            for batch_data in train_loader:
                optimizer.zero_grad()

                # Forward pass
                temporal_features = batch_data['temporal'].to(self.device)
                spatial_features = batch_data['spatial'].to(self.device)
                edge_index = batch_data['edge_index'].to(self.device)
                edge_weight = batch_data['edge_weight'].to(self.device)

                regression_pred, classification_pred, recovery_pred = self.model(
                    temporal_features, spatial_features, edge_index, edge_weight
                )

                # Calculate losses
                reg_loss = mse_loss(regression_pred, batch_data['regression_target'].to(self.device))
                cls_loss = bce_loss(classification_pred, batch_data['classification_target'].to(self.device))
                rec_loss = mse_loss(recovery_pred, batch_data['recovery_target'].to(self.device))

                total_loss = reg_loss + cls_loss + 0.5 * rec_loss

                # Backward pass
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss += total_loss.item()

                # Calculate accuracy for classification
                cls_pred_binary = (torch.sigmoid(classification_pred) > 0.5).float()
                accuracy = (cls_pred_binary == batch_data['classification_target'].to(self.device)).float().mean()
                train_acc += accuracy.item()

                num_batches += 1

            # Validation phase
            val_loss, val_acc = self._validate(val_loader, mse_loss, bce_loss)

            # Update learning rate
            scheduler.step(val_loss)

            # Record history
            avg_train_loss = train_loss / num_batches
            avg_train_acc = train_acc / num_batches

            self.training_history['train_loss'].append(avg_train_loss)
            self.training_history['val_loss'].append(val_loss)
            self.training_history['train_acc'].append(avg_train_acc)
            self.training_history['val_acc'].append(val_acc)

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(self.model.state_dict(), 'best_outage_model.pth')
            else:
                patience_counter += 1

            if patience_counter >= 20:
                print(f"Early stopping at epoch {epoch}")
                break

            if epoch % 10 == 0:
                print(f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={val_loss:.4f}, "
                      f"Train Acc={avg_train_acc:.4f}, Val Acc={val_acc:.4f}")

        # Load best model
        self.model.load_state_dict(torch.load('best_outage_model.pth', map_location=self.device))
        print("✅ Training completed!")

    def _validate(self, val_loader, mse_loss, bce_loss):
        """Validation step"""
        self.model.eval()
        val_loss = 0
        val_acc = 0
        num_batches = 0

        with torch.no_grad():
            for batch_data in val_loader:
                temporal_features = batch_data['temporal'].to(self.device)
                spatial_features = batch_data['spatial'].to(self.device)
                edge_index = batch_data['edge_index'].to(self.device)
                edge_weight = batch_data['edge_weight'].to(self.device)

                regression_pred, classification_pred, recovery_pred = self.model(
                    temporal_features, spatial_features, edge_index, edge_weight
                )

                reg_loss = mse_loss(regression_pred, batch_data['regression_target'].to(self.device))
                cls_loss = bce_loss(classification_pred, batch_data['classification_target'].to(self.device))
                rec_loss = mse_loss(recovery_pred, batch_data['recovery_target'].to(self.device))

                total_loss = reg_loss + cls_loss + 0.5 * rec_loss
                val_loss += total_loss.item()

                cls_pred_binary = (torch.sigmoid(classification_pred) > 0.5).float()
                accuracy = (cls_pred_binary == batch_data['classification_target'].to(self.device)).float().mean()
                val_acc += accuracy.item()

                num_batches += 1

        return val_loss / num_batches, val_acc / num_batches

    def plot_training_history(self):
        """Plot training history"""
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Loss plot
        axes[0].plot(self.training_history['train_loss'], label='Train Loss')
        axes[0].plot(self.training_history['val_loss'], label='Validation Loss')
        axes[0].set_title('Training Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()

        # Accuracy plot
        axes[1].plot(self.training_history['train_acc'], label='Train Accuracy')
        axes[1].plot(self.training_history['val_acc'], label='Validation Accuracy')
        axes[1].set_title('Classification Accuracy')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()

        plt.tight_layout()
        plt.show()

# ============================
# 4. PREDICTION AND RISK ASSESSMENT
# ============================

class OutageRiskAssessor:
    def __init__(self, model, processor, device):
        self.model = model.to(device)
        self.processor = processor
        self.device = device

    def predict_outage_risk(self, current_data, weather_forecast):
        """Predict outage risk for next 12 hours"""
        self.model.eval()

        with torch.no_grad():
            # Prepare input data
            temporal_features = torch.tensor(current_data['temporal'], dtype=torch.float32).to(self.device)
            spatial_features = torch.tensor(current_data['spatial'], dtype=torch.float32).to(self.device)
            edge_index = current_data['edge_index'].to(self.device)
            edge_weight = current_data['edge_weight'].to(self.device)

            # Get predictions
            regression_pred, classification_pred, recovery_pred = self.model(
                temporal_features, spatial_features, edge_index, edge_weight
            )

            # Convert to probabilities
            extreme_event_prob = torch.sigmoid(classification_pred).cpu().numpy()
            risk_scores = regression_pred[:, :, 0].cpu().numpy()  # Risk score predictions
            outage_rates = regression_pred[:, :, 1].cpu().numpy()  # Outage rate predictions
            recovery_times = recovery_pred.cpu().numpy()

            return {
                'extreme_event_probability': extreme_event_prob,
                'risk_scores': risk_scores,
                'predicted_outage_rates': outage_rates,
                'predicted_recovery_times': recovery_times
            }

    def generate_risk_report(self, predictions, county_names):
        """Generate comprehensive risk assessment report"""
        report = {
            'high_risk_counties': [],
            'extreme_event_alerts': [],
            'recovery_time_estimates': {},
            'summary_statistics': {}
        }

        for i, county in enumerate(county_names):
            max_risk = predictions['risk_scores'][i].max()
            max_extreme_prob = predictions['extreme_event_probability'][i].max()
            avg_recovery_time = predictions['predicted_recovery_times'][i].mean()

            # High risk counties (risk score > 0.7)
            if max_risk > 0.7:
                report['high_risk_counties'].append({
                    'county': county,
                    'risk_score': max_risk,
                    'extreme_probability': max_extreme_prob
                })

            # Extreme event alerts (probability > 0.5)
            if max_extreme_prob > 0.5:
                report['extreme_event_alerts'].append({
                    'county': county,
                    'probability': max_extreme_prob,
                    'estimated_recovery_hours': avg_recovery_time
                })

            report['recovery_time_estimates'][county] = avg_recovery_time

        # Summary statistics
        report['summary_statistics'] = {
            'counties_at_high_risk': len(report['high_risk_counties']),
            'extreme_event_alerts': len(report['extreme_event_alerts']),
            'average_risk_score': np.mean([pred['risk_score'] for pred in report['high_risk_counties']] or [0]),
            'average_recovery_time': np.mean(list(report['recovery_time_estimates'].values()))
        }

        return report

# ============================
# 5. MAIN EXECUTION PIPELINE
# ============================

def create_data_loader(sequences, targets_reg, targets_cls, targets_rec,
                      edge_index, edge_weight, batch_size=32):
    """Create data loader for training"""

    # Convert to proper format for batching
    dataset = []
    for i in range(len(sequences)):
        # Create individual data points
        temporal_seq = torch.tensor(sequences[i], dtype=torch.float32)

        data_dict = {
            'temporal': temporal_seq,
            'spatial': temporal_seq,  # Use same features for spatial
            'regression_target': torch.tensor(targets_reg[i], dtype=torch.float32),
            'classification_target': torch.tensor(targets_cls[i], dtype=torch.float32),
            'recovery_target': torch.tensor(targets_rec[i], dtype=torch.float32),
        }
        dataset.append(data_dict)

    # Custom collate function to handle graph data
    def custom_collate(batch):
        # Stack temporal and spatial features
        temporal_batch = torch.stack([item['temporal'] for item in batch])
        spatial_batch = torch.stack([item['spatial'] for item in batch])

        # Stack targets
        reg_targets = torch.stack([item['regression_target'] for item in batch])
        cls_targets = torch.stack([item['classification_target'] for item in batch])
        rec_targets = torch.stack([item['recovery_target'] for item in batch])

        return {
            'temporal': temporal_batch,
            'spatial': spatial_batch,
            'regression_target': reg_targets,
            'classification_target': cls_targets,
            'recovery_target': rec_targets,
            'edge_index': edge_index,
            'edge_weight': edge_weight
        }

    return DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate)


def main():
    """Main execution pipeline"""
    print("🔥 SPATIO-TEMPORAL POWER OUTAGE PREDICTION MODEL")
    print("=" * 60)

    # 🔽 Set up device (Apple GPU > CUDA > CPU)
    device = get_device()

    # 1. Load and preprocess data
    processor = PowerOutageDataProcessor()

    # Use your optimized dataset
    df = processor.load_and_preprocess_data('../data/processed/optimized_michigan_outages_2019.csv')

    # 2. Create county graph
    edge_index, edge_weight, county_coords = processor.create_county_graph(df)

    # 3. Prepare sequences
    sequences, targets_reg, targets_cls, targets_rec = processor.prepare_sequences(df)

    # 4. Split data
    train_idx, val_idx = train_test_split(
        range(len(sequences)), test_size=0.2, random_state=42
    )

    train_sequences = sequences[train_idx]
    train_targets_reg = targets_reg[train_idx]
    train_targets_cls = targets_cls[train_idx]
    train_targets_rec = targets_rec[train_idx]

    val_sequences = sequences[val_idx]
    val_targets_reg = targets_reg[val_idx]
    val_targets_cls = targets_cls[val_idx]
    val_targets_rec = targets_rec[val_idx]

    # 5. Create data loaders
    train_loader = create_data_loader(
        train_sequences, train_targets_reg, train_targets_cls, train_targets_rec,
        edge_index, edge_weight, batch_size=16
    )

    val_loader = create_data_loader(
        val_sequences, val_targets_reg, val_targets_cls, val_targets_rec,
        edge_index, edge_weight, batch_size=16
    )

    # 6. Initialize model
    input_dim = sequences.shape[2]  # Number of features
    num_counties = len(processor.county_to_idx)

    model = SpatioTemporalOutagePredictor(
        input_dim=input_dim,
        spatial_dim=64,
        temporal_dim=64,
        output_dim=32,
        num_counties=num_counties,
        prediction_horizon=12
    )

    print(f"🎯 Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

    # 7. Train model
    trainer = OutageModelTrainer(model, device)
    trainer.train_model(train_loader, val_loader, epochs=50, lr=0.001)

    # 8. Plot training history
    trainer.plot_training_history()

    # =====================
    # 🔽 Save the model and processor
    # =====================
    print("💾 Saving model and data processor...")
    torch.save(model, 'outage_full_model.pt')  # Saves the full model

    with open('outage_data_processor.pkl', 'wb') as f:
        pickle.dump(processor, f)  # Saves the processor object

    print("✅ Model saved as 'outage_full_model.pt'")
    print("✅ Processor saved as 'outage_data_processor.pkl'")

    # 9. Create risk assessor
    risk_assessor = OutageRiskAssessor(model, processor, device)

    print("✅ Model training completed!")
    print("🚀 Ready for real-time outage prediction and risk assessment!")

    # 10. Example prediction (using validation data)
    sample_data = {
        'temporal': val_sequences[0:1],
        'spatial': val_sequences[0:1],
        'edge_index': edge_index,
        'edge_weight': edge_weight
    }

    predictions = risk_assessor.predict_outage_risk(sample_data, None)
    county_names = list(processor.idx_to_county.values())[:1]

    risk_report = risk_assessor.generate_risk_report(predictions, county_names)

    print("\n📊 SAMPLE RISK ASSESSMENT REPORT:")
    print(f"High risk counties: {risk_report['summary_statistics']['counties_at_high_risk']}")
    print(f"Extreme event alerts: {risk_report['summary_statistics']['extreme_event_alerts']}")
    print(f"Average risk score: {risk_report['summary_statistics']['average_risk_score']:.3f}")

    return model, processor, risk_assessor

if __name__ == "__main__":
    model, processor, risk_assessor = main()